# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


### Part 1)



- The CSV file is available at https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv
- The file contains a list of famous people and a brief overview.
- The goal of part 1) is to ...
  1. Pick one person from the list ( the reference person ) and output 10 other people who's overview are "closest" to the reference person in a Natural Language Processing sense
  1. Also output the sentiment of the overview of the reference person



### Part 2)



- For the same reference person that you chose in Part 1), use the Wikipedia API to access the whole content of the reference person's Wikipedia page.
- The goal of Part 2) is to ...
  1. Print out the text of the Wikipedia article for the reference person
  1. Determine the sentiment of the text of the Wikipedia page for the reference person
  1. Collect the text of the Wikipedia pages from the 10 nearest neighbors from Part 1)
  1. Determine the nearness ranking of these 10 people to your reference person based on their entire Wikipedia page
  1. Compare, i.e. plot,  the nearest ranking from Step 1) with the Wikipedia page nearness ranking.  A difference of the rank is one means of comparison.



### Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


In [35]:
%%capture output
#install Wikipedia API
!pip3 install wikipedia-api


In [36]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

import nltk
# nltk.download('omw-1.4')
nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

import wikipediaapi

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
!curl -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv | wc -l

42786


In [3]:
%%bash
curl -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv |
head -1 |
tr , '\n' |
cat -n


     1	URI
     2	name
     3	text


In [4]:
%%bash
curl -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv |
head -2 |
tail -1 |
tr , '\n' |
cat -n


     1	<http://dbpedia.org/resource/Digby_Morrell>
     2	Digby Morrell
     3	digby morrell born 10 october 1979 is a former australian rules footballer who played with the kangaroos and carlton in the australian football league aflfrom western australia morrell played his early senior football for west perth his 44game senior career for the falcons spanned 19982000 and he was the clubs leading goalkicker in 2000 at the age of 21 morrell was recruited to the australian football league by the kangaroos football club with its third round selection in the 2001 afl rookie draft as a forward he twice kicked five goals during his time with the kangaroos the first was in a losing cause against sydney in 2002 and the other the following season in a drawn game against brisbaneafter the 2003 season morrell was traded along with david teague to the carlton football club in exchange for corey mckernan he played 32 games for the blues before being delisted at the end of 2005 he continued to play v

In [5]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'

In [6]:
data = pd.read_csv(url)

In [7]:
data

,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...
...,...,...,...
42781,<http://dbpedia.org/resource/Motoaki_Takenouchi>,Motoaki Takenouchi,motoaki takenouchi born july 8 1967 saitama pr...
42782,<http://dbpedia.org/resource/Alan_Judge_(footb...,"Alan Judge (footballer, born 1960)",alan graham judge born 14 may 1960 is a retire...
42783,<http://dbpedia.org/resource/Eduardo_Lara>,Eduardo Lara,eduardo lara lozano born 4 september 1959 in c...
42784,<http://dbpedia.org/resource/Tatiana_Faberg%C3...,Tatiana Faberg%C3%A9,tatiana faberg is an author and faberg scholar...


In [8]:
data_c = data.copy()

## Part 1

In [9]:
data_c.drop( columns = ['URI'])

,name,text
0,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,G-Enka,henry krvits born 30 december 1974 in tallinn ...
...,...,...
42781,Motoaki Takenouchi,motoaki takenouchi born july 8 1967 saitama pr...
42782,"Alan Judge (footballer, born 1960)",alan graham judge born 14 may 1960 is a retire...
42783,Eduardo Lara,eduardo lara lozano born 4 september 1959 in c...
42784,Tatiana Faberg%C3%A9,tatiana faberg is an author and faberg scholar...


In [10]:
# I want to singularize "born"
# later I will compare who has is closer is age by trying to figure out how to single out birth date


In [11]:
# i would like to just begin with a smaller range of data to start light and build

In [12]:
data_c = data_c.drop(range(21, len(data_c)))

In [13]:
data_c

,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...
5,<http://dbpedia.org/resource/Sam_Henderson>,Sam Henderson,sam henderson born october 18 1969 is an ameri...
6,<http://dbpedia.org/resource/Aaron_LaCrate>,Aaron LaCrate,aaron lacrate is an american music producer re...
7,<http://dbpedia.org/resource/Trevor_Ferguson>,Trevor Ferguson,trevor ferguson aka john farrow born 11 novemb...
8,<http://dbpedia.org/resource/Grant_Nelson>,Grant Nelson,grant nelson born 27 april 1971 in london also...
9,<http://dbpedia.org/resource/Cathy_Caruth>,Cathy Caruth,cathy caruth born 1955 is frank h t rhodes pro...


In [28]:
print(data_c.loc[0, 'text'])

digby morrell born 10 october 1979 is a former australian rules footballer who played with the kangaroos and carlton in the australian football league aflfrom western australia morrell played his early senior football for west perth his 44game senior career for the falcons spanned 19982000 and he was the clubs leading goalkicker in 2000 at the age of 21 morrell was recruited to the australian football league by the kangaroos football club with its third round selection in the 2001 afl rookie draft as a forward he twice kicked five goals during his time with the kangaroos the first was in a losing cause against sydney in 2002 and the other the following season in a drawn game against brisbaneafter the 2003 season morrell was traded along with david teague to the carlton football club in exchange for corey mckernan he played 32 games for the blues before being delisted at the end of 2005 he continued to play victorian football league vfl football with the northern bullants carltons vflaf

## cleaning

In [40]:
# 1. Access index 0 using .loc[index, 'column_name']
sentence_0_tb = TextBlob(data_c.loc[0, 'text'])
sentence_0_singular = [x.singularize() for x in sentence_0_tb.words]
sentence_0_clean = ' '.join(sentence_0_singular)

In [39]:
sentence_0_clean

'digby morrell born 10 october 1979 is a former australian rule footballer who played with the kangaroo and carlton in the australian football league aflfrom western australium morrell played hi early senior football for west perth hi 44game senior career for the falcon spanned 19982000 and he wa the club leading goalkicker in 2000 at the age of 21 morrell wa recruited to the australian football league by the kangaroo football club with it third round selection in the 2001 afl rookie draft a a forward he twice kicked five goal during hi time with the kangaroo the first wa in a losing cause against sydney in 2002 and the other the following season in a drawn game against brisbaneafter the 2003 season morrell wa traded along with david teague to the carlton football club in exchange for corey mckernan he played 32 game for the blue before being delisted at the end of 2005 he continued to play victorian football league vfl football with the northern bullant carlton vflaffiliate in 2006 an

In [41]:
# must find a way to loop and clean the 20 rows that I have to minimize the number of nique characters that are practically the same
# Create an empty list to store our cleaned sentences
cleaned_list = []

# Loop through every row in your light dataset
for text in data_c['text']:
    # 1. Turn the text row into a TextBlob object
    tb = TextBlob(text)

    # 2. Singularize every word in that blob
    singular_words = [x.singularize() for x in tb.words]

    # 3. Join the words back together into a single string
    clean_sentence = ' '.join(singular_words)

    # 4. Append the clean string to our list
    cleaned_list.append(clean_sentence)

# Add the list as a new column in your DataFrame
data_c['cleaned_text'] = cleaned_list

# View the result
data_c[['name', 'cleaned_text']].head()

,name,cleaned_text
0,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,G-Enka,henry krvit born 30 december 1974 in tallinn b...


In [45]:
# 1. Initialize the CountVectorizer (ignores common stop words)
vectorizer = CountVectorizer(stop_words='english')

# 2. Fit and transform your newly cleaned text column
bag_of_words = vectorizer.fit_transform(data_c['cleaned_text'])

# 3. Convert it into a clean, readable DataFrame to see the counts
bow_df = pd.DataFrame(
    bag_of_words.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=data_c['name']
)

# View the word count table
bow_df.head()

,10,1000,102,11,12,1406,17,173182,17th,18,...,yale,year,york,yorksince,young,youngest,zarathustra,zelazny,ziet,zwigoff
name,,,,,,,,,,,,,,,,,,,,,
Digby Morrell,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Alfred J. Lewy,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Harpdog Brown,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
Franz Rottensteiner,0,0,0,0,0,0,0,0,0,1,...,0,4,1,0,0,0,0,1,0,0
G-Enka,0,0,0,0,0,0,0,0,0,1,...,0,2,0,0,0,0,0,0,0,0


AttributeError: 'DataFrame' object has no attribute 'toarray'

## TF-IDF

In [46]:
# 1. Initialize the TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# 2. Fit and transform your cleaned text
tfidf_matrix = tfidf_vectorizer.fit_transform(data_c['cleaned_text'])

# 3. Convert it into a readable DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=data_c['name']
)

# View the TF-IDF table
tfidf_df.head()

,10,1000,102,11,12,1406,17,173182,17th,18,...,yale,year,york,yorksince,young,youngest,zarathustra,zelazny,ziet,zwigoff
name,,,,,,,,,,,,,,,,,,,,,
Digby Morrell,0.043715,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
Alfred J. Lewy,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
Harpdog Brown,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.083337,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
Franz Rottensteiner,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.046996,...,0.0,0.131622,0.039951,0.0,0.0,0.0,0.0,0.059039,0.0,0.0
G-Enka,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.051197,...,0.0,0.071695,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


In [48]:
tf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=data_c['name']
)

# Check the shape of the DataFrame
tf_df.shape

(21, 1954)

In [49]:
tf_df

,10,1000,102,11,12,1406,17,173182,17th,18,...,yale,year,york,yorksince,young,youngest,zarathustra,zelazny,ziet,zwigoff
name,,,,,,,,,,,,,,,,,,,,,
Digby Morrell,0.043715,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Alfred J. Lewy,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Harpdog Brown,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.0000,0.083337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Franz Rottensteiner,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.046996,...,0.0000,0.131622,0.039951,0.000000,0.000000,0.000000,0.000000,0.059039,0.000000,0.000000
G-Enka,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.051197,...,0.0000,0.071695,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Sam Henderson,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.048578,...,0.0000,0.034013,0.123887,0.061027,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Aaron LaCrate,0.027488,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.0000,0.000000,0.025468,0.000000,0.033146,0.037637,0.000000,0.000000,0.000000,0.000000
Trevor Ferguson,0.000000,0.041802,0.000000,0.041802,0.000000,0.000000,0.041802,0.00000,0.000000,0.000000,...,0.0000,0.023299,0.084860,0.000000,0.000000,0.000000,0.041802,0.000000,0.041802,0.000000
Grant Nelson,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [52]:
df = pd.DataFrame(
    tfidf_matrix[3].T.todense(),
    index=tfidf_vectorizer.get_feature_names_out(),
    columns=["TF-IDF"]
)

# Sort words by highest TF-IDF score
df = df.sort_values('TF-IDF', ascending=False)

# Display top words
df.head(10)

,TF-IDF
fiction,0.328969
science,0.239703
lem,0.236157
rottensteiner,0.177117
book,0.172474
number,0.155982
author,0.140987
year,0.131622
1975,0.118078
austrium,0.118078
